In [42]:
ensure_package <- function(pkg) {
  # Quietly installs if not installed, and then loads the library
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg, dependencies = TRUE)
  }
  library(pkg, character.only = TRUE)
}

# Advanced *R*
This is a jupyter notebook meant to go through the [Advanced R](https://adv-r.hadley.nz/introduction.html) Markdown Book.
## Introduction


In [43]:
ensure_package("lobstr") # Apparently for digging into memory management

x <- c(1, 2, 3) # Creates a vector and binds it to the name "x"
y <- x # `y` is now a reference to `x` (does not create a copy)

# Exercises

# 1
a <- 1:10 # A range from 1 to 10
b <- a # A reference to a
c <- b # A reference to a
d <- 1:10 # A new range

# 2
print(paste(
  obj_addr(mean),
  obj_addr(base::mean),
  obj_addr(get("mean")),
  obj_addr(evalq(mean)),
  obj_addr(match.fun("mean"))
))
# All end up pointing to the same function

# 3
# For `read.csv`, set the check.names parameter to FALSE in order to supress
# conversion of non-syntactic names to syntactic ones. This could be used if
# the names follow some pattern in which some would be syntactic and some
# wouldn't, making conversion unpredictable.

# 4
# It either prepends 'X' or appends '.'

# 5
# Variable names may not start with '.' and then a number

[1] "0x2009aaa1990 0x2009aaa1990 0x2009aaa1990 0x2009aaa1990 0x2009aaa1990"


## 2.3 Copy-On-Modify

Objects in R are **immutable**, so when you modify them, as in
```R
x <- c(1,2,3)
x[[1]] <- 1
```
R actually copies `x` to a new location. Thus,
```R
x <- c(1, 2, 3)
y <- x

y[[3]] <- 4
x
#> [1] 1 2 3
```
### `tracemem()`
The function `base::tracemem()` will bind to a variable, and will tell you whenever it moves as a result of mutation.

In [44]:
# 2.3.6 Exercises

# 1. The range is not bound to anything, so `tracemem()` is transient

# 2. I'm guessing it has two steps, one for modifying the datatype and
#    another for modifying the value

# 3. See my scrap paper

a <- 1:10
b <- list(a, a)
c <- list(b, a, 1:10)

ref(a)
ref(b)
ref(c)

# 4. a list containing the range from one to ten as its only element is
#    bound to `x`, and then copied, with the second element being the original
#    list.

[1:0x2009ffb8208] <int> 

█ [1:0x200d1033788] <list> 
├─[2:0x2009ffb8208] <int> 
└─[2:0x2009ffb8208] 

█ [1:0x200d1049458] <list> 
├─█ [2:0x200d1033788] <list> 
│ ├─[3:0x2009ffb8208] <int> 
│ └─[3:0x2009ffb8208] 
├─[3:0x2009ffb8208] 
└─[4:0x200d10aa240] <int> 

## Vectors

There are two types of vectors, **atomic vectors** (where all elements must have the same type) and **lists** (where elements may differ in type).

`NULL` is not a vector, but is closely related and often serves as a 0-length vector.

Vectors may have attributes (think of them as a named list of arbitrary metadata)
- The `dimension` attribute turns vectors into matrices and arrays
- The `class` attribute powers the S3 object system
    - Important S3 Vectors: factors, date/times, data frames, tibbles

### Creating Vectors
The `c()` function (short for combine) creates a vector, as in
```R
x <- c(1,2,3)
```
Using the `c()` function on a set of vectors flattens them:
```R
c(c(1, 2), c(3, 4))
#> [1] 1 2 3 4
```

### Attributes

In [45]:
# Getting and setting attributes

a <- 1:3
attr(a, "x") <- "abcdef"
attr(a, "x") # returns 'abcdef'
attr(a, "y") <- 4:6

str(attributes(a))

[1] "abcdef"

List of 2
 $ x: chr "abcdef"
 $ y: int [1:3] 4 5 6


In [46]:
# NAMING VECTORS

# When creating it
x <- c(a = 1, b = 2, c = 3)
x

# By assigning a character vector to names()
x <- 1:3
names(x) <- c("a", "b", "c")
x

# Inline using setNames()
x <- setNames(1:3, c("a", "b", "c"))
x

# AVOID using attr(x, "names"), as it requires more typing and is less readable

a b c 
1 2 3

a b c 
1 2 3

a b c 
1 2 3

### Dimensions

Setting the dimension of a vector allows it to behave like a 2D matrix or $n$-dimensional array.

The following are roughly equivalent methods for different types of vectors depending on dimension:
|Vector|Matrix|Array|
|--- |--- |--- |
|names()|rownames(), colnames()|dimnames()|
|length()|nrow(), ncol()|dim()|
|c()|rbind(), cbind()|abind::abind()|
|—|t()|aperm()|
|is.null(dim(x))|is.matrix()|is.array()|

In [47]:
# DIMENSIONS

# Using the arguments of the matrix() function
x <- matrix(1:6, nrow = 2, ncol = 3)
x

# Using a vector argument of the array() function
y <- array(1:12, c(2, 3, 2))
print(y)

# Modifying a vector in place by setting dim()
z <- 1:6
dim(z) <- c(3, 2)
z

1,3,5
2,4,6


, , 1

     [,1] [,2] [,3]
[1,]    1    3    5
[2,]    2    4    6

, , 2

     [,1] [,2] [,3]
[1,]    7    9   11
[2,]    8   10   12



1,4
2,5
3,6


In [48]:
# 3.3 Exercises

# 1. setNames(x, vec) simply binds vec to names(x)

# 2. It returns NULL. You would use NROW and NCOL if you want to treat a simple
#    vector as a column vector

# 3. x1 has a sort of depth of 5, x2 is a 5-column row vector, x3 is a 5-row
#    column vector. Simply using 1:5 can be treated as a column vector as in x3,
#    but calling nrow() will only return a number on x3

### S3 Atomic Vectors
Important S3 objects:
- Factors: factors are used to store categorical data. They are built on top of an integer vector with two attributes: a `class`, "factor", which makes it behave differently, and `levels`, which defines the set of allowed vectors.

In [49]:
x <- factor(c("a", "b", "b", "a"))
x

[1] a b b a
Levels: a b

- Dates: Date vectors are built on top of double vectors. They have class "Date" and no other attributes:

In [50]:
today <- Sys.Date()

today
typeof(today)
attributes(today)

[1] "2024-06-11"

[1] "double"

$class
[1] "Date"

- Date-times: date-time information is stored in Base R in two ways, POSIXct and POSIXlt
  - POSIX stands for Portable Operating System Interface (a family of cross-platform standards)
  - "ct" stands for calendar time (it's just seconds since epoch)
  - "lt" stands for local time (it's a list of time differences which is stupid and bad because obviously.)

In [51]:
now_ct <- as.POSIXct("2018-08-01 22:00", tz = "EST")
now_ct

attributes(now_ct)

# Returns now_ct but in the JST time zone
structure(now_ct, tzone = "Asia/Tokyo")

[1] "2018-08-01 22:00:00 EST"

$class
[1] "POSIXct" "POSIXt" 

$tzone
[1] "EST"

[1] "2018-08-02 12:00:00 JST"

## Lists
A step up in complexity, each element can be any type, and they aren't really there, they're just references to other objects (which can actually be any type)

In [52]:
# Construct a list with list()
l1 <- list(
  1:3,
  "a",
  c(TRUE, FALSE, TRUE),
  c(2.3, 5.9)
)

In [53]:
list(list(1, 2), c(3, 4)) # Creates a list containing a list and a vector
c(list(1, 2), c(3, 4)) # Combine into one list

[[1]]
[[1]][[1]]
[1] 1

[[1]][[2]]
[1] 2


[[2]]
[1] 3 4

[[1]]
[1] 1

[[2]]
[1] 2

[[3]]
[1] 3

[[4]]
[1] 4

You can turn a list into an atomic vector with `unlist()`. Don't. If it's a list, it's a list for a reason.

### Matrices and Arrays
You can (if so inclined) create list-matrices and list-arrays using the `dim` attribute.

In [54]:
l <- list(1:3, "a", TRUE, 1.0)
dim(l) <- c(2, 2)
l

"1, 2, 3",TRUE
a,1


### Data frames
If you're doing data analysis in R, you're definitely using data frames. A data frame is a named list of vectors with attributes for (column) `names`, `row.names` and its class, "data.frame"

In [55]:
df <- data.frame(x = 4:6, y = letters[1:3])
typeof(df)

attributes(df)

[1] "list"

$names
[1] "x" "y"

$class
[1] "data.frame"

$row.names
[1] 1 2 3

### Tibbles

Tibbles are like data frames, but they do less and complain more

In [56]:
ensure_package("tibble")

df <- tibble(x = 4:6, y = letters[1:3])

typeof(df)
attributes(df)

[1] "list"

$class
[1] "tbl_df"     "tbl"        "data.frame"

$row.names
[1] 1 2 3

$names
[1] "x" "y"

### Data Frames v.s. Tibbles
1. Tibbles do not coerce their input (data frames convert vectors of characters to factors, unless `stringsAsFactors` is set to `FALSE`)
2. Tibbles do not rename non-syntactic names (data frames do, unless `check.names` is set to `FALSE`)
3. Both datatypes recycle short inputs, but tibbles only recycle inputs of length 1

In [57]:
ensure_package("dplyr")
print(dplyr::starwars)

# tibbles are pwettyyyyy

# A tibble: 87 × 14
   name     height  mass hair_color skin_color eye_color birth_year sex   gender
   <chr>     <int> <dbl> <chr>      <chr>      <chr>          <dbl> <chr> <chr> 
 1 Luke Sk…    172    77 blond      fair       blue            19   male  mascu…
 2 C-3PO       167    75 NA         gold       yellow         112   none  mascu…
 3 R2-D2        96    32 NA         white, bl… red             33   none  mascu…
 4 Darth V…    202   136 none       white      yellow          41.9 male  mascu…
 5 Leia Or…    150    49 brown      light      brown           19   fema… femin…
 6 Owen La…    178   120 brown, gr… light      blue            52   male  mascu…
 7 Beru Wh…    165    75 brown      light      blue            47   fema… femin…
 8 R5-D4        97    32 NA         white, red red             NA   none  mascu…
 9 Biggs D…    183    84 black      light      brown           24   male  mascu…
10 Obi-Wan…    182    77 auburn, w… fair       blue-gray       57   male  mascu…
# ℹ 77 m

## Subsetting
Easy to get started, difficult to master.
- There are **six** ways to subset atomic vectors
- There are three subsetting operators: `[[`, `[`, and `$`
- Subsetting operators' behavior depends on the type of vector
- Subsetting may be combined with assignment

In [58]:
# 4.2 Selecting Multiple Elements Using [

x <- c(2.1, 4.2, 3.3, 5.4) # An atomic vector

# Positive integers return elements at the specified position(s)
x[c(3, 1)]

# Duplicate indices duplicate outputs
x[c(1, 1)]

# Real numbers are truncated to integers
x[c(2.1, 2.9)]

# Negative integers exclude elements at the specified position(s) (No mixing)
x[-c(3, 1)]

# Logical vectors select elements where the corresponding logical value is TRUE
x[c(TRUE, TRUE, FALSE, FALSE)]
x[x > 3]

# If x and y are different lenghts, x[y] is determined using the recycling rules
# Ew.

# Missing values in the index yield missing values in the output
x[c(TRUE, FALSE, NA, FALSE)]


[1] 3.3 2.1

[1] 2.1 2.1

[1] 4.2 4.2

[1] 4.2 5.4

[1] 2.1 4.2

[1] 4.2 3.3 5.4

[1] 2.1  NA

- **Nothing** returns the original vector (not useful for 1D vectors, but is for others)

In [59]:
x[]

[1] 2.1 4.2 3.3 5.4

- **Zero** Returns a zero-length vector.

In [60]:
x[0]

numeric(0)

- Use **character vectors** to return elements with matching names.

In [61]:
y <- setNames(x, letters[1:4])

y[c("d", "c", "a")]

# Repeated Indices
y[c("a", "a", "a")]

# When subsetting with [, names are always matched exactly
z <- c(abc = 1, def = 2)
z[c("a", "d")]

d   c   a 
5.4 3.3 2.1

a   a   a 
2.1 2.1 2.1

<NA> <NA> 
  NA   NA

Factors are not treated specially (They use the underlying integer vector, not the character levels)

### Lists
Subsetting lists are the same as subsetting vectors. `[` returns a list. `[[` and `$` pull elements out of the list.

### Matrices and Arrays
You can subset higher-dimensional structures in three ways:
- With multiple vectors
- With a single vector
- With a matrix

In [62]:
a <- matrix(1:9, nrow = 3)
colnames(a) <- c("A", "B", "C")

# Getting the values matching the cart. product of the vectors (r, c)
a[1:2, c("A", "B")]

# Blank Subsetting keeps all the rows/columns
a[1:2, ]

# Using a logical vector
a[c(TRUE, FALSE, TRUE), c("B", "A")]

A,B
1,4
2,5


A,B,C
1,4,7
2,5,8


B,A
4,1
6,3


By default, `[` simplifies to the lowest possible dimensionality.

In [63]:
a[1, ] # Returns a named vector
a[1, 1] # Returns a scalar

A B C 
1 4 7

A 
1

In [64]:
(vals <- outer(1:5, 1:5, FUN = "paste", sep = ", "))

vals[c(4, 15)]

"1, 1","1, 2","1, 3","1, 4","1, 5"
"2, 1","2, 2","2, 3","2, 4","2, 5"
"3, 1","3, 2","3, 3","3, 4","3, 5"
"4, 1","4, 2","4, 3","4, 4","4, 5"
"5, 1","5, 2","5, 3","5, 4","5, 5"


[1] "4, 1" "5, 3"

In [65]:
select <- matrix(ncol = 2, byrow = TRUE, c(
  1, 1,
  3, 1,
  2, 4
))

vals[select]

[1] "1, 1" "3, 1" "2, 4"

### Data Frames and Tibbles
- When subsetting with a single index, they behave like lists and index the columns
- When subsetting with two indices, they behave like matrices, so `df[1:3, ]` selects the first three *rows* (and all columns)

In [66]:
(df <- data.frame(x = 1:3, y = 3:1, z = letters[1:3]))

df[df$x == 2, ] # Uses a logical vector to select the second row

df[c(1, 3), ] # Selects the first and third row

# Selecting Columns from a data frame
# Like a list
df[c("x", "y")]

# List a matrix
df[, c("x", "y")]

x,y,z
<int>,<int>,<chr>
1,3,a
2,2,b
3,1,c


,x,y,z
,<int>,<int>,<chr>
2,2,2,b


,x,y,z
,<int>,<int>,<chr>
1,1,3,a
3,3,1,c


x,y
<int>,<int>
1,3
2,2
3,1


x,y
<int>,<int>
1,3
2,2
3,1


The behavior of these subsetting methods differs when selecting a single column:

In [67]:
# Matrix subsetting simplifies to a vector by default
str(df[, "x"])

# List subsetting does not
str(df["x"])

 int [1:3] 1 2 3
'data.frame':	3 obs. of  1 variable:
 $ x: int  1 2 3


By default, subsetting a matrix with a single number, single name, or a logical vector containing only onw `TRUE` will simplify the returned output, i.e. it will return an object with lower dimensionality. To preserve the original dimensionality, use `drop = FALSE`

In [68]:
# MATRICES

a <- matrix(1:6, ncol = 2)

str(a[, 1]) # simplifies to a 1D vector
str(a[, 1, drop = FALSE]) # retains 2D

 int [1:3] 1 2 3
 int [1:3, 1] 1 2 3


In [69]:
# DATA FRAMES

df <- data.frame(x = 1:4, y = letters[1:4])

str(df[, "x"]) # Simplifies to a 1D vector
str(df[, "x", drop = FALSE]) # Does not simplify, stays a data frame

 int [1:4] 1 2 3 4
'data.frame':	4 obs. of  1 variable:
 $ x: int  1 2 3 4


THIS IS WHY TIBBLES ARE GOOD!!!! (of course no one will ever adopt them because they were invented in 2015 and every legacy package that you need to make anything work is based on the inane behavior of dataframes)

In [70]:
mtcars[mtcars$cyl == 4, ]
mtcars[-(1:4), ]
mtcars[mtcars$cyl <= 5, ]
mtcars[mtcars$cyl == 4 | mtcars$cyl == 6, ]

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
Merc 240D,24.4,4,146.7,62,3.69,3.190,20.00,1,0,4,2
Merc 230,22.8,4,140.8,95,3.92,3.150,22.90,1,0,4,2
Fiat 128,32.4,4,78.7,66,4.08,2.200,19.47,1,1,4,1
Honda Civic,30.4,4,75.7,52,4.93,1.615,18.52,1,1,4,2
Toyota Corolla,33.9,4,71.1,65,4.22,1.835,19.90,1,1,4,1
Toyota Corona,21.5,4,120.1,97,3.70,2.465,20.01,1,0,3,1
Fiat X1-9,27.3,4,79.0,66,4.08,1.935,18.90,1,1,4,1
Porsche 914-2,26.0,4,120.3,91,4.43,2.140,16.70,0,1,5,2


,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2
Valiant,18.1,6,225.0,105,2.76,3.460,20.22,1,0,3,1
Duster 360,14.3,8,360.0,245,3.21,3.570,15.84,0,0,3,4
Merc 240D,24.4,4,146.7,62,3.69,3.190,20.00,1,0,4,2
Merc 230,22.8,4,140.8,95,3.92,3.150,22.90,1,0,4,2
Merc 280,19.2,6,167.6,123,3.92,3.440,18.30,1,0,4,4
Merc 280C,17.8,6,167.6,123,3.92,3.440,18.90,1,0,4,4
Merc 450SE,16.4,8,275.8,180,3.07,4.070,17.40,0,0,3,3
Merc 450SL,17.3,8,275.8,180,3.07,3.730,17.60,0,0,3,3


,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
Merc 240D,24.4,4,146.7,62,3.69,3.190,20.00,1,0,4,2
Merc 230,22.8,4,140.8,95,3.92,3.150,22.90,1,0,4,2
Fiat 128,32.4,4,78.7,66,4.08,2.200,19.47,1,1,4,1
Honda Civic,30.4,4,75.7,52,4.93,1.615,18.52,1,1,4,2
Toyota Corolla,33.9,4,71.1,65,4.22,1.835,19.90,1,1,4,1
Toyota Corona,21.5,4,120.1,97,3.70,2.465,20.01,1,0,3,1
Fiat X1-9,27.3,4,79.0,66,4.08,1.935,18.90,1,1,4,1
Porsche 914-2,26.0,4,120.3,91,4.43,2.140,16.70,0,1,5,2


,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
Valiant,18.1,6,225.0,105,2.76,3.460,20.22,1,0,3,1
Merc 240D,24.4,4,146.7,62,3.69,3.190,20.00,1,0,4,2
Merc 230,22.8,4,140.8,95,3.92,3.150,22.90,1,0,4,2
Merc 280,19.2,6,167.6,123,3.92,3.440,18.30,1,0,4,4
Merc 280C,17.8,6,167.6,123,3.92,3.440,18.90,1,0,4,4


## Contol Flow

There are two primary methods of control flow, choices and loops.
- Choices: Things like `if` and `switch` statements (conditionals)
- Loops: Things like `while` and `for` loops

### Choices

An if statement is written as follows:
```R
if (condition) true_action
if (condition) true_action else false_action
```
Typically, the actions are compound statements contained within `{`
```R
grade <- function(x) {
  if (x > 90) {
    "A"
  } else if (x > 80) {
    "B"
  } else if (x > 50) {
    "C"
  } else {
    "F"
  }
}
```
If there is no `else` statement, `if` invisibly returns `NULL`. *Advanced R* points that since `paste()` and `c()` drop `NULL` values, this can be useful.

R does not have a well-developed sense of truthyness or falsyness. It interprets non-zero numbers as `TRUE` and zero as `FALSE`. All others throw an error. Also, by default it only evaluates the first element of a vector. If it is a zero-length vector, you're fucked.

#### Vectorized If

Since `if` is a mid function tbh, what do you do if you need to evaluate the truth of a vector of values? You use `ifelse()`, a function (not a primitive like `if`).

In [71]:
x <- 1:10

# ifelse (logical, yes, no)
ifelse (x %% 5 == 0, "XXX", as.character(x))

[1] "1"   "2"   "3"   "4"   "XXX" "6"   "7"   "8"   "9"   "XXX"

In [72]:
# Multiple conditions on one object

x_option <- function(x) {
  if (x == "a") {
    "option 1"
  } else if (x == "b") {
    "option 2"
  } else if (x == "c") {
    "option 3"
  } else {
    stop("Invalid `x` value")
  }
}
x_option("b")

# Equivalently:

x_option <- function(x) {
  switch(x,
    a = "option 1",
    b = "option 2",
    c = "option 3",
    stop("Invalid `x` value")
  )
}
x_option("c")

[1] "option 2"

[1] "option 3"

If multiple inputs should correspond to one output, you can leave the right-hand-side of `=` empty and the input will "fall through" to the next value (this can cascade)

In [73]:
get_number_of_legs <- function(x) {
  switch(x,
    cow = ,
    horse = ,
    cat = ,
    dog = 4,
    human = ,
    chicken = 2,
    plant = 0,
    stop("Unknown input")
  )
}

get_number_of_legs("cow")

[1] 4

## Loops

`for` loops are used to iterate over items in a vector, and have the following structure:
```R
for (item in vector) perform_action
```
For each item in `vector`, `perform_action` is called once, updatin the value of `item` each time.

`for` assigns the `item` to the current environment, overwriting any existing variable with the same name:

In [74]:
i <- 100
for (i in 1:3) {}
i

[1] 3

There are two ways to end a `for` loop early:
- `next` skips to the next iteration
- `break` exits the loop entirely

### Common Pitfalls

When generating data, be sure to preallocate the output container. Otherwise the loop will be very slow.

In [75]:
means <- c(1, 50, 20)
out <- vector("list", length(means))
for (i in seq_along(means)) {
  out[[i]] <- rnorm(10, means[[i]])
}

Beware of iterating over S3 vectors, as loops typically strip any attributes:

In [76]:
# Bad (just returns the underlying numbers):
xs <- as.Date(c("2020-01-01", "2010-01-01"))
for (x in xs) {
  print(x)
}

# Better:
for (i in seq_along(xs)) {
  print(xs[[i]])
}

[1] 18262
[1] 14610
[1] "2020-01-01"
[1] "2010-01-01"


## Functions

Functions in R have three parts:
- The `formals()`, the list of arguments that control how you call the function
- The `body()`, the code inside the function
- The `environment()`, the data structure that determines how the function finds the values associated with the names

In [77]:
f <- function(x, y) {
  # A comment
  x + y
}
formals(f)
body(f)
environment(f)

$x


$y



{
    x + y
}

<environment: R_GlobalEnv>

The `srcref` attribute of an R function contains the full source code (inlcuding comments)

In [78]:
attr(f, "srcref")

function(x, y) {
  # A comment
  x + y
}

R functions are just objects, which make them "first-class functions". Unlike in some other languages, you use the `function()` function to create a function, and bind that to a name with `<-`.

You can also use anonymous functions.

In [79]:
ensure_package("magrittr")

objs <- mget(ls("package:base", all = TRUE), inherits = TRUE)
funs <- Filter(is.function, objs)
# Filter(function(fun) )
# sort()
for (i in seq_along(funs)) {
  funs[[i]] <- length(formals(funs[[i]]))
}
sort(unlist(funs), decreasing = TRUE)[1:1]

funs %>%
  unlist() %>%
  sort(decreasing = TRUE) %>%
  print()

scan 
  22

                              scan                             source 
                                22                                 17 
                    format.default                            formatC 
                                16                                 15 
                           library                   merge.data.frame 
                                13                                 13 
                         prettyNum                            system2 
                                13                                 12 
                            system                  all.equal.numeric 
                                11                                 10 
                     print.default                               save 
                                10                                 10 
                     withAutoprint                            .pretty 
                                10                                  9 
      

In [80]:
f2 <- function(x = z) {
  z <- 100
  x
}
f2()

[1] 100

# The Tidyverse n Stuff

In [84]:
library(nycflights13)

print(flights)

# A tibble: 336,776 × 19
    year month   day dep_time sched_dep_time dep_delay arr_time sched_arr_time
   <int> <int> <int>    <int>          <int>     <dbl>    <int>          <int>
 1  2013     1     1      517            515         2      830            819
 2  2013     1     1      533            529         4      850            830
 3  2013     1     1      542            540         2      923            850
 4  2013     1     1      544            545        -1     1004           1022
 5  2013     1     1      554            600        -6      812            837
 6  2013     1     1      554            558        -4      740            728
 7  2013     1     1      555            600        -5      913            854
 8  2013     1     1      557            600        -3      709            723
 9  2013     1     1      557            600        -3      838            846
10  2013     1     1      558            600        -2      753            745
# ℹ 336,766 more rows
# ℹ 1